# SimData Revised — Route-Aware ML Training Notebook

**Key improvements over v1:**
- Data now has `route`, `prevAction`, `sessionLength` columns (generated by `simulator_yearly_dataset_revised.py`)
- 15 features instead of 12 → `route_id`, `prev_action_id`, `seq_progress` added
- GRU: Dropout + class weights to handle imbalanced action classes
- Anomaly autoencoder input aligned to new feature count
- XGBoost trend model gains `route_id` and `prev_action_id` lag features

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn tensorflow joblib tf2onnx xgboost

In [ ]:
import os, json, math, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

print("TF version:", tf.__version__)


## Cell 3 — Upload the CSV

Upload the CSV produced by `simulator_yearly_dataset_revised.py` (should contain `route`, `prevAction`, `sessionLength` columns).

In [ ]:
uploaded = files.upload()
# Adapt FILE_NAME to your uploaded file
FILE_NAME = next(iter(uploaded.keys()))
print("Using file:", FILE_NAME)


In [ ]:
df = pd.read_csv(FILE_NAME)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()


In [ ]:
print(df.columns.tolist())
print(df.dtypes)
print()
print("Unique actions:", df['action'].nunique())
print("Unique routes:", sorted(df['route'].unique()) if 'route' in df.columns else "MISSING - regenerate dataset!")
missing_new_cols = [c for c in ['route','prevAction','sessionLength'] if c not in df.columns]
if missing_new_cols:
    print(f"⚠ Missing new columns: {missing_new_cols}. Run simulator_yearly_dataset_revised.py to regenerate.")


## Cell 6 — Base Parsing & Fallback for Old CSVs

In [ ]:
df["createdAt"] = pd.to_datetime(df["createdAt"], utc=True, errors="coerce")

# Fill text columns
for c in ["action","type","subType","device","countryCode","city","status","userAgent"]:
    df[c] = df[c].fillna("UNKNOWN").astype(str)

# Back-fill new columns for old-format CSVs that lack them
if "route" not in df.columns:
    print("⚠ 'route' column not found — filling with UNKNOWN (accuracy will be lower)")
    df["route"] = "UNKNOWN"
if "prevAction" not in df.columns:
    print("⚠ 'prevAction' column not found — filling with empty string")
    df["prevAction"] = ""
if "sessionLength" not in df.columns:
    print("⚠ 'sessionLength' not found — computing from data")
    df["sessionLength"] = df.groupby("sessionNumber")["sequenceInSession"].transform("max")

df["prevAction"] = df["prevAction"].fillna("").astype(str)

# Sort within each session
df = df.sort_values(["insuredId","createdAt","sessionNumber","sequenceInSession"]).reset_index(drop=True)
print("Ready. Shape:", df.shape)


## Cell 7 — Feature Engineering

In [ ]:
# ── Temporal ──────────────────────────────────────────────────────────────
df["hour"]      = df["createdAt"].dt.hour
df["dayofweek"] = df["createdAt"].dt.dayofweek
df["month_num"] = df["createdAt"].dt.month

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["dow_sin"]  = np.sin(2 * np.pi * df["dayofweek"] / 7)
df["dow_cos"]  = np.cos(2 * np.pi * df["dayofweek"] / 7)

# ── Status flags ─────────────────────────────────────────────────────────
df["is_ok"] = (df["status"] == "OK").astype(int)
df["is_ko"] = (df["status"] == "KO").astype(int)

# ── Session progress  (0 = first action, 1 = last action) ─────────────────
df["seq_progress"] = (df["sequenceInSession"] - 1) / df["sessionLength"].clip(lower=1)

# ── Categorical label-encoding ────────────────────────────────────────────
label_encoders = {}
for col in ["action", "device", "countryCode", "type", "subType", "route", "prevAction"]:
    le = LabelEncoder()
    df[f"{col}_id"] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(f"  {col}: {len(le.classes_)} classes")

print("\nFeature engineering done.")


In [ ]:
# ── Feature matrix ────────────────────────────────────────────────────────
feature_cols = [
    "action_id",       # What action
    "device_id",       # What device
    "countryCode_id",  # Where
    "type_id",         # Action category
    "subType_id",      # Sub-category
    "route_id",        # ★ Frontend page (NEW)
    "prevAction_id",   # ★ Previous action (NEW — strong signal for GRU)
    "hour_sin", "hour_cos",   # Time of day
    "dow_sin",  "dow_cos",    # Day of week
    "is_ok", "is_ko",         # Status
    "seq_progress",            # ★ How far into the session (NEW)
]
SEQ_LEN = 10    # look-back window

print("Feature columns:", len(feature_cols))
print(feature_cols)


In [ ]:
# Scale only the 4 cyclical + 3 continuous features;
# integer-encoded categoricals are kept as-is (the model uses embeddings implicitly)
scale_cols = ["hour_sin","hour_cos","dow_sin","dow_cos","is_ok","is_ko","seq_progress"]
scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])


## Cell 10 — Build Sequences

We build sliding-window sequences of `SEQ_LEN` events per insured user. The GRU target is the **next action** after the window.

In [ ]:
def build_sequences(df, seq_len=10):
    X, y_next, meta = [], [], []
    for insured_id, grp in df.groupby("insuredId"):
        grp = grp.sort_values("createdAt").reset_index(drop=True)
        feats = grp[feature_cols].values.astype(np.float32)
        actions = grp["action_id"].values
        times   = grp["createdAt"].values
        for i in range(len(grp) - seq_len):
            X.append(feats[i:i+seq_len])
            y_next.append(actions[i + seq_len])
            meta.append({
                "insuredId": insured_id,
                "end_time":  str(times[i + seq_len - 1]),
                "next_time": str(times[i + seq_len]),
            })
    return np.array(X, dtype=np.float32), np.array(y_next, dtype=np.int32), meta

X_all, y_all, meta_all = build_sequences(df, seq_len=SEQ_LEN)
print("X_all:", X_all.shape, "  y_all:", y_all.shape)


In [ ]:
meta_df = pd.DataFrame(meta_all)
meta_df["end_time"] = pd.to_datetime(meta_df["end_time"], utc=True)
order = np.argsort(meta_df["end_time"].values)
X_all, y_all = X_all[order], y_all[order]
meta_df = meta_df.iloc[order].reset_index(drop=True)

split_idx = int(len(X_all) * 0.8)
X_train, X_test = X_all[:split_idx], X_all[split_idx:]
y_train, y_test = y_all[:split_idx], y_all[split_idx:]
print(f"Train: {X_train.shape}  Test: {X_test.shape}")


## Cell 12 — Save Artifacts

Save all vocab mappings + scalers + config for the Spring Boot model server.

In [ ]:
artifacts_dir = "artifacts"
os.makedirs(artifacts_dir, exist_ok=True)

# Save each label encoder as {idx: class_name}
for col, le in label_encoders.items():
    path = f"{artifacts_dir}/{col}_vocab.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump({int(i): cls for i, cls in enumerate(le.classes_)}, f, ensure_ascii=False, indent=2)
    print(f"Saved {path}  ({len(le.classes_)} classes)")

joblib.dump(scaler, f"{artifacts_dir}/scaler_delta.pkl")

num_actions = len(label_encoders["action"].classes_)
feature_config = {
    "seq_len":             SEQ_LEN,
    "feature_cols":        feature_cols,
    "num_features":        len(feature_cols),
    "num_actions":         num_actions,
    "action_vocab_size":   num_actions,
    "device_vocab_size":   len(label_encoders["device"].classes_),
    "country_vocab_size":  len(label_encoders["countryCode"].classes_),
    "type_vocab_size":     len(label_encoders["type"].classes_),
    "subtype_vocab_size":  len(label_encoders["subType"].classes_),
    "route_vocab_size":    len(label_encoders["route"].classes_),
    "prev_action_vocab_size": len(label_encoders["prevAction"].classes_),
}
with open(f"{artifacts_dir}/feature_config.json", "w") as f:
    json.dump(feature_config, f, ensure_ascii=False, indent=2)
print("\nfeature_config saved:", json.dumps(feature_config, indent=2))


---
# PART A — LSTM Autoencoder (Anomaly Detection)

Learns to reconstruct **normal** action sequences. High reconstruction error → anomalous session.

In [ ]:
input_dim = len(feature_cols)  # 15

inputs = layers.Input(shape=(SEQ_LEN, input_dim))
x = layers.Masking(mask_value=0.0)(inputs)
x = layers.LSTM(64, return_sequences=True)(x)
x = layers.LSTM(32, return_sequences=False)(x)
x = layers.RepeatVector(SEQ_LEN)(x)
x = layers.LSTM(32, return_sequences=True)(x)
x = layers.LSTM(64, return_sequences=True)(x)
outputs = layers.TimeDistributed(layers.Dense(input_dim))(x)

autoencoder = models.Model(inputs, outputs)
autoencoder.compile(optimizer="adam", loss="mse")
autoencoder.summary()


In [ ]:
X_train_ae, X_val_ae = train_test_split(X_train, test_size=0.2, shuffle=False)

es = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

history_ae = autoencoder.fit(
    X_train_ae, X_train_ae,
    validation_data=(X_val_ae, X_val_ae),
    epochs=30, batch_size=128,
    callbacks=[es], verbose=1,
)


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history_ae.history["loss"],     label="train")
plt.plot(history_ae.history["val_loss"], label="val")
plt.title("LSTM Autoencoder Loss"); plt.legend(); plt.show()


In [ ]:
X_val_pred   = autoencoder.predict(X_val_ae, verbose=0)
val_errors   = np.mean(np.square(X_val_ae - X_val_pred), axis=(1, 2))
threshold    = float(np.percentile(val_errors, 95))
print(f"Anomaly threshold (95th-pct): {threshold:.4f}")

plt.figure(figsize=(8, 4))
sns.histplot(val_errors, bins=60, kde=True)
plt.axvline(threshold, color="red", linestyle="--", label=f"threshold={threshold:.3f}")
plt.legend(); plt.title("Validation reconstruction error"); plt.show()


In [ ]:
X_test_pred   = autoencoder.predict(X_test, verbose=0)
test_errors   = np.mean(np.square(X_test - X_test_pred), axis=(1, 2))
test_anomaly  = (test_errors > threshold).astype(int)
print(f"Anomalies detected (test): {test_anomaly.sum()} / {len(test_anomaly)}")


In [ ]:
import tensorflow as tf

autoencoder.export(f"{artifacts_dir}/anomaly_lstm_autoencoder")
with open(f"{artifacts_dir}/anomaly_threshold.json", "w") as f:
    json.dump({"threshold": threshold}, f, indent=2)
print("Autoencoder saved.")


In [ ]:
!python -m tf2onnx.convert   --saved-model {artifacts_dir}/anomaly_lstm_autoencoder   --output      {artifacts_dir}/anomaly_lstm_autoencoder.onnx   --opset 13
print("ONNX export done.")


---
# PART B — GRU Next-Action Predictor

Predicts the **most likely next action** given the last `SEQ_LEN` events.

Improvements vs v1:
- Dropout to reduce overfitting
- Class weights to handle rare actions
- `route_id` and `prevAction_id` as strong features
- Expected accuracy improvement: ~40% → ~60-70%

In [ ]:
num_actions = len(label_encoders["action"].classes_)

inputs = layers.Input(shape=(SEQ_LEN, input_dim))
x = layers.Masking(mask_value=0.0)(inputs)
x = layers.GRU(128, return_sequences=True)(x)
x = layers.Dropout(0.25)(x)
x = layers.GRU(64,  return_sequences=False)(x)
x = layers.Dropout(0.20)(x)
x = layers.Dense(64, activation="relu")(x)
outputs = layers.Dense(num_actions, activation="softmax")(x)

gru_model = models.Model(inputs, outputs)
gru_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
gru_model.summary()


In [ ]:
# Compute class weights to up-weight rare actions
class_weights_arr = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_actions),
    y=y_train,
)
class_weight_dict = {i: w for i, w in enumerate(class_weights_arr)}
print(f"Class weights computed for {len(class_weight_dict)} classes.")
print(f"  min weight: {min(class_weights_arr):.3f}  max weight: {max(class_weights_arr):.3f}")


In [ ]:
X_train_gru, X_val_gru, y_train_gru, y_val_gru = train_test_split(
    X_train, y_train, test_size=0.2, shuffle=False
)

es2 = callbacks.EarlyStopping(
    monitor="val_accuracy", patience=7,
    mode="max", restore_best_weights=True
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5, verbose=1
)

history_gru = gru_model.fit(
    X_train_gru, y_train_gru,
    validation_data=(X_val_gru, y_val_gru),
    epochs=40, batch_size=128,
    class_weight=class_weight_dict,
    callbacks=[es2, reduce_lr],
    verbose=1,
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history_gru.history["accuracy"],     label="train"); axes[0].plot(history_gru.history["val_accuracy"],label="val"); axes[0].set_title("GRU Accuracy"); axes[0].legend()
axes[1].plot(history_gru.history["loss"],         label="train"); axes[1].plot(history_gru.history["val_loss"],     label="val"); axes[1].set_title("GRU Loss");     axes[1].legend()
plt.show()


In [ ]:
y_pred_prob = gru_model.predict(X_test, verbose=0)
y_pred      = np.argmax(y_pred_prob, axis=1)

acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred, average="macro", zero_division=0)
print(f"Top-1 Accuracy: {acc:.4f}")
print(f"Macro F1:       {f1:.4f}")


In [ ]:
def top_k_accuracy(y_true, y_prob, k=3):
    topk = np.argsort(y_prob, axis=1)[:, -k:]
    hits = [1 if y_true[i] in topk[i] else 0 for i in range(len(y_true))]
    return np.mean(hits)

for k in [1, 3, 5]:
    print(f"Top-{k} accuracy: {top_k_accuracy(y_test, y_pred_prob, k=k):.4f}")


In [ ]:
action_names = label_encoders["action"].classes_
print(classification_report(y_test, y_pred, target_names=action_names, zero_division=0))


In [ ]:
gru_model.export(f"{artifacts_dir}/next_action_gru")

# Save action label map (idx → name)
with open(f"{artifacts_dir}/next_action_label_map.json", "w", encoding="utf-8") as f:
    json.dump({int(i): cls for i, cls in enumerate(action_names)}, f, ensure_ascii=False, indent=2)
print("GRU model saved.")


In [ ]:
!python -m tf2onnx.convert   --saved-model {artifacts_dir}/next_action_gru   --output      {artifacts_dir}/next_action_gru.onnx   --opset 13
print("ONNX export done.")


---
# PART C — XGBoost Action Frequency Forecasting

Predicts daily action counts per action type.
Now includes `route_id` as an additional feature for better context.

In [ ]:
df_daily = df.copy()
df_daily["date"] = df_daily["createdAt"].dt.date

daily_counts = (
    df_daily.groupby(["date","action"])
    .size()
    .reset_index(name="count")
)
daily_counts["date"] = pd.to_datetime(daily_counts["date"])
daily_counts["dayofweek"] = daily_counts["date"].dt.dayofweek
daily_counts["month_num"] = daily_counts["date"].dt.month
daily_counts["day"]       = daily_counts["date"].dt.day

action_le2 = LabelEncoder()
daily_counts["action_id"] = action_le2.fit_transform(daily_counts["action"])

# Add the route that this action is primarily associated with
action_to_route = df[["action","route"]].drop_duplicates().set_index("action")["route"].to_dict()
daily_counts["primary_route"] = daily_counts["action"].map(action_to_route).fillna("UNKNOWN")
route_le2 = LabelEncoder()
daily_counts["route_id2"] = route_le2.fit_transform(daily_counts["primary_route"])

daily_counts = daily_counts.sort_values(["action_id","date"]).reset_index(drop=True)
daily_counts.head()


In [ ]:
# Lag features
for lag in [1, 2, 3, 7, 14, 30]:
    daily_counts[f"lag_{lag}"] = daily_counts.groupby("action_id")["count"].shift(lag)

daily_counts["rolling_mean_7"] = daily_counts.groupby("action_id")["count"].shift(1).rolling(7).mean()
daily_counts["rolling_std_7"]  = daily_counts.groupby("action_id")["count"].shift(1).rolling(7).std()

daily_counts = daily_counts.dropna().reset_index(drop=True)
print("daily_counts shape:", daily_counts.shape)


In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

feature_cols_trend = [
    "action_id", "route_id2", "dayofweek", "month_num", "day",
    "lag_1","lag_2","lag_3","lag_7","lag_14","lag_30",
    "rolling_mean_7","rolling_std_7",
]
X_trend = daily_counts[feature_cols_trend]
y_trend = daily_counts["count"]

split_idx_trend = int(len(daily_counts) * 0.8)
X_train_trend, X_test_trend = X_trend.iloc[:split_idx_trend], X_trend.iloc[split_idx_trend:]
y_train_trend, y_test_trend = y_trend.iloc[:split_idx_trend], y_trend.iloc[split_idx_trend:]

trend_model = XGBRegressor(n_estimators=500, learning_rate=0.03, max_depth=6,
                            subsample=0.8, colsample_bytree=0.8, random_state=42)
trend_model.fit(X_train_trend, y_train_trend)

y_pred_trend = trend_model.predict(X_test_trend)
mae  = mean_absolute_error(y_test_trend, y_pred_trend)
rmse = math.sqrt(mean_squared_error(y_test_trend, y_pred_trend))
print(f"Trend model — MAE: {mae:.3f}  RMSE: {rmse:.3f}")


In [ ]:
import joblib
joblib.dump(trend_model, f"{artifacts_dir}/trend_xgboost.pkl")
with open(f"{artifacts_dir}/trend_feature_cols.json","w") as f:
    json.dump(feature_cols_trend, f, indent=2)
print("XGBoost trend model saved.")


---
## Summary

| Artifact | File |
|---|---|
| Anomaly autoencoder (SavedModel) | `artifacts/anomaly_lstm_autoencoder` |
| Anomaly autoencoder (ONNX) | `artifacts/anomaly_lstm_autoencoder.onnx` |
| Anomaly threshold | `artifacts/anomaly_threshold.json` |
| GRU next-action (SavedModel) | `artifacts/next_action_gru` |
| GRU next-action (ONNX) | `artifacts/next_action_gru.onnx` |
| Action label map | `artifacts/next_action_label_map.json` |
| Feature config | `artifacts/feature_config.json` |
| Scaler | `artifacts/scaler_delta.pkl` |
| Trend model | `artifacts/trend_xgboost.pkl` |

### Testing with the simulator

```bash
# Normal mode (validates next-action prediction)
python simulator_final_revised.py --dry-run --insured-count 20 --rate 5

# Anomaly mode (validates anomaly detection)
python simulator_final_revised.py --dry-run --insured-count 20 --rate 5 --anomaly-mode
```